# RoadScan — fixed-prediction re-scoring audit
Test localization, fragment grouping and class-confusion hypotheses without training or new inference. Reuse ONLY the full-frame predictions and unchanged ground truth saved in notebook 11. Do not use crop predictions or open test data. Record the protocol before scoring. Relaxed/merged metrics diagnose sensitivity; they do not replace the official detection metrics or demonstrate improved deployment performance. Label conventions remain a hypothesis, not an established cause.


## 1. Register the protocol and verify cached inputs
Run on CPU with Accelerator=None. The cache uses the retained notebook 10 checkpoint, 640 square inputs, confidence floor 0.001, class-aware NMS IoU 0.70, and a saved cap of 100 detections/image. It was generated in batches of two during the notebook 11 localization audit. The earlier Ultralytics table used batch four and cap 300: reproduce the cached COCO baseline and report this difference rather than claiming an identical protocol. A prediction cap cannot be undone by re-scoring. Hash inputs and verify complete coverage of all 2,977 development images. No model is loaded.


In [1]:
import sys, subprocess, json, hashlib, time, io, contextlib
from pathlib import Path
from collections import defaultdict, Counter
subprocess.run([sys.executable,'-m','pip','install','-q','pycocotools'],check=True,timeout=120)
import numpy as np
import pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from IPython.display import display, Markdown
BASE=Path('/kaggle/input/notebooks/rimasabunasser/11-roadscan-sampling-comparison/balance_pilot')
OUT=Path('/kaggle/working/rescoring_audit');OUT.mkdir(exist_ok=True)
CLASSES=['D00','D10','D20','D40']
protocol=dict(source_notebook_version=359330902,weights='notebook 10 selected checkpoint; notebook 11 unchanged',imgsz=640,conf_floor=0.001,nms_iou=0.70,saved_max_det=100,inference_batch=2,bootstrap_replicates=1000,bootstrap_seed=20261017,merge_gap_fraction=0.02,merge_score='maximum member confidence',merge_link='connected components of ORIGINAL same-class rectangles; normalized x/y gaps each <=0.02',thresholds=[0.1,0.3,0.5],criterion_D00_merged_China=0.30,criterion_control_D20_max_abs_change=0.03,criterion_D00_AP10_China=0.30,criterion_pooled_class_gap=0.10)
(OUT/'PROTOCOL.json').write_text(json.dumps(protocol,indent=2))
start=time.monotonic();datasets={};input_hashes={}
for domain,expected in [('in_domain_dev',1000),('china_dev',1977)]:
    paths=[BASE/'localization_check'/f'{domain}_ground_truth.json',BASE/'localization_check'/f'{domain}_full_frame_predictions.json']
    gt,pred=[json.loads(p.read_text()) for p in paths]
    for p in paths:input_hashes[str(p.relative_to(BASE))]=hashlib.sha256(p.read_bytes()).hexdigest()
    assert len(gt['images'])==expected and len({x['id'] for x in gt['images']})==expected
    assert [c['name'] for c in gt['categories']]==CLASSES
    ids={im['id'] for im in gt['images']};assert all(p['image_id'] in ids for p in pred)
    assert all(len(p['bbox'])==4 and np.isfinite(p['bbox']).all() and p['bbox'][2]>0 and p['bbox'][3]>0 and p['category_id'] in range(4) and 0.001-1e-8<=p['score']<=1 for p in pred)
    counts=Counter(p['image_id'] for p in pred)
    assert max(counts.values(),default=0)<=100
    print(domain,'images:',expected,'GT:',len(gt['annotations']),'predictions:',len(pred),'images_at_saved_cap:',sum(v==100 for v in counts.values()),flush=True)
    datasets[domain]=(gt,pred)
(OUT/'INPUT_SHA256.json').write_text(json.dumps(input_hashes,indent=2))


in_domain_dev images: 1000 GT: 1260 predictions: 30346 images_at_saved_cap: 60
china_dev images: 1977 GT: 4650 predictions: 73023 images_at_saved_cap: 24


510

## 2. Count support and recover monitor composition
Report reference boxes and positive images for every country/class, including background images. Use saved filenames to identify countries; retain all images in each group. Recover the exact 400-image monitor from its saved manifest. Similar D10 AP across domains is not proof of perfect generalization: its absolute AP remains limited and uncertainty matters. The cache contains no epoch-zero predictions, but earlier notebooks contain evaluations of predecessor checkpoints; distinguish these from the first pilot epoch.


In [2]:
support=[];group_images={};monitor_rows=[]
for domain,(gt,pred) in datasets.items():
    ims={im['id']:im for im in gt['images']};ann=defaultdict(list)
    for a in gt['annotations']:ann[a['image_id']].append(a)
    groups={domain:sorted(ims)}
    if domain=='in_domain_dev':
        for country in ['Czech','India','Japan']:groups[country]=[i for i,im in ims.items() if im['file_name'].startswith(country+'_')]
        assert sum(len(groups[c]) for c in ['Czech','India','Japan'])==1000
    for group,image_ids in groups.items():
        group_images[group]=(domain,image_ids)
        for cls,name in enumerate(CLASSES):
            support.append(dict(group=group,class_name=name,images=len(image_ids),positive_images=sum(any(a['category_id']==cls for a in ann[i]) for i in image_ids),instances=sum(sum(a['category_id']==cls for a in ann[i]) for i in image_ids),background_images=sum(not ann[i] for i in image_ids)))
    filenames={im['file_name']:im['id'] for im in gt['images']}
    manifest=pd.read_csv(BASE/'monitor_manifest.csv')
    part=manifest[manifest.domain==domain]
    selected=[filenames[Path(p).name] for p in part.path]
    assert len(selected)==200
    monitor_rows.append(dict(domain=domain,images=len(selected),backgrounds=sum(not ann[i] for i in selected),instances=sum(len(ann[i]) for i in selected)))
support=pd.DataFrame(support);support.to_csv(OUT/'support.csv',index=False);display(support)
monitor=pd.DataFrame(monitor_rows);monitor.to_csv(OUT/'monitor_composition.csv',index=False);display(monitor)
assert monitor.images.sum()==400


,group,class_name,images,positive_images,instances,background_images
0,in_domain_dev,D00,1000,250,348,358
1,in_domain_dev,D10,1000,153,250,358
2,in_domain_dev,D20,1000,349,451,358
3,in_domain_dev,D40,1000,126,211,358
4,Czech,D00,202,50,62,133
5,Czech,D10,202,13,17,133
6,Czech,D20,202,10,14,133
7,Czech,D40,202,13,17,133
8,India,D00,124,15,17,67
9,India,D10,124,1,1,67


,domain,images,backgrounds,instances
0,in_domain_dev,200,65,256
1,china_dev,200,6,459


## 3. Implement and verify one-to-one matching
For each image, order detections by confidence and greedily match each to the highest-IoU unmatched eligible reference. Compute COCO-style 101-point interpolated AP. Validate per-class AP50 and AP10/AP30 against pycocotools before using the implementation for resampling. All references have iscrowd=0 and positive area; no ignored instances are introduced.

For fragment merging, connect ORIGINAL boxes of the same class when horizontal and vertical edge gaps, divided by image width/height, are each <=2%. Use transitive connected components, their enclosing rectangle and maximum prediction confidence. Apply the identical rule independently to GT and predictions for D00/D10 and the D20 control; leave D40 unchanged. This may merge distinct nearby defects, and low-confidence prediction chains may create large components. Save component diagnostics; any score change cannot prove label correctness.

For class confusion, compare pooled class-aware AP50 with pooled class-agnostic AP50 on identical detections and the same reference denominator, for all classes and cracks (D00–D20) separately. Do not compare pooled AP directly with macro-average class AP.


In [3]:
def xyxy(records):
    a=np.asarray([r['bbox'] for r in records],dtype=float).reshape(-1,4)
    if len(a):a[:,2:]+=a[:,:2]
    return a

def box_iou(a,b):
    lo=np.maximum(a[:2],b[:,:2]);hi=np.minimum(a[2:],b[:,2:])
    inter=np.maximum(hi-lo,0).prod(axis=1)
    return inter/np.maximum((a[2:]-a[:2]).prod()+(b[:,2:]-b[:,:2]).prod(axis=1)-inter,1e-12)

def merge_records(records,image,prediction):
    result=[];log=[]
    for cls in range(4):
        items=[r for r in records if r['category_id']==cls]
        if cls==3:result.extend(dict(r) for r in items);continue
        boxes=xyxy(items);parent=list(range(len(items)))
        def find(i):
            while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]
            return i
        if len(boxes):
            gap_x=np.maximum(np.maximum(boxes[:,None,0]-boxes[None,:,2],boxes[None,:,0]-boxes[:,None,2]),0)/image['width']
            gap_y=np.maximum(np.maximum(boxes[:,None,1]-boxes[None,:,3],boxes[None,:,1]-boxes[:,None,3]),0)/image['height']
            for i,j in zip(*np.where(np.triu((gap_x<=0.02)&(gap_y<=0.02),k=1))):parent[find(j)]=find(i)
        components=defaultdict(list)
        for i in range(len(items)):components[find(i)].append(i)
        for members in components.values():
            selected=boxes[members];lo=selected[:,:2].min(axis=0);hi=selected[:,2:].max(axis=0)
            item=dict(items[members[0]]);item['bbox']=[*lo,*(hi-lo)]
            if prediction:item['score']=max(items[i]['score'] for i in members)
            else:item.update(area=float(np.prod(hi-lo)),iscrowd=0)
            result.append(item)
            log.append(dict(image_id=image['id'],class_name=CLASSES[cls],kind='prediction' if prediction else 'GT',members=len(members),width_fraction=float((hi[0]-lo[0])/image['width']),height_fraction=float((hi[1]-lo[1])/image['height'])))
    return result,log

def make_metric(gt,pred,ids,classes,threshold,agnostic=False):
    positions={image_id:k for k,image_id in enumerate(ids)};g=defaultdict(list);p=defaultdict(list)
    for a in gt:
        if a['image_id'] in positions and a['category_id'] in classes:g[a['image_id']].append(a)
    for a in pred:
        if a['image_id'] in positions and a['category_id'] in classes:p[a['image_id']].append(a)
    ngt=np.zeros(len(ids),dtype=int);scores=[];tp=[];image_index=[]
    for image_id in ids:
        refs=g[image_id];boxes=xyxy(refs);labels=np.array([r['category_id'] for r in refs]);used=set();ngt[positions[image_id]]=len(refs)
        for detection in sorted(p[image_id],key=lambda r:-r['score']):
            b=xyxy([detection])[0];overlap=box_iou(b,boxes) if len(boxes) else np.array([])
            eligible=[j for j in range(len(refs)) if j not in used and (agnostic or labels[j]==detection['category_id']) and overlap[j]>=threshold]
            matched=bool(eligible)
            if matched:used.add(max(eligible,key=lambda j:(overlap[j],j)))
            scores.append(detection['score']);tp.append(matched);image_index.append(positions[image_id])
    order=np.argsort(-np.array(scores),kind='stable')
    tp=np.array(tp,dtype=bool)[order];image_index=np.array(image_index,dtype=int)[order]
    # Trailing false positives cannot increase the interpolated precision envelope.
    if tp.any():cut=np.flatnonzero(tp)[-1]+1;tp=tp[:cut];image_index=image_index[:cut]
    else:tp=tp[:0];image_index=image_index[:0]
    return dict(tp=tp,index=image_index,ngt=ngt)

def average_precision(metric,weights=None):
    if weights is None:weights=np.ones(len(metric['ngt']),dtype=int)
    count=float(metric['ngt']@weights)
    if count==0:return float('nan')
    w=weights[metric['index']];active=w>0
    if not active.any():return 0.0
    tp=metric['tp'][active];w=w[active]
    true=np.cumsum(w*tp);false=np.cumsum(w*(~tp))
    recall=true/count;precision=true/(true+false+np.spacing(1))
    precision=np.maximum.accumulate(precision[::-1])[::-1]
    index=np.searchsorted(recall,np.linspace(0,1,101),side='left')
    values=np.zeros(101);valid=index<len(precision);values[valid]=precision[index[valid]]
    return float(values.mean())

# Hand-checked cases: perfect localization, wrong class, missing detection and fragment union.
_example_gt=[dict(image_id=1,category_id=0,bbox=[0.,0.,10.,10.])]
_example_pred=[dict(image_id=1,category_id=0,bbox=[0.,0.,10.,10.],score=0.9)]
assert np.isclose(average_precision(make_metric(_example_gt,_example_pred,[1],[0],0.5)),1.0)
assert average_precision(make_metric(_example_gt,[],[1],[0],0.5))==0.0
_wrong=[dict(_example_pred[0],category_id=1)]
assert average_precision(make_metric(_example_gt,_wrong,[1],[0,1],0.5))==0.0
assert np.isclose(average_precision(make_metric(_example_gt,_wrong,[1],[0,1],0.5,True)),1.0)
_frag=[dict(image_id=1,category_id=0,bbox=[0.,0.,10.,10.],score=.8),dict(image_id=1,category_id=0,bbox=[11.,0.,10.,10.],score=.6)]
_joined,_=merge_records(_frag,dict(id=1,width=100,height=100),True)
assert len(_joined)==1 and _joined[0]['bbox']==[0.,0.,21.,10.] and _joined[0]['score']==.8
print('Hand-checked matching and merge cases passed.')

merged={};merge_logs=[]
for domain,(gt,pred) in datasets.items():
    grouped_gt=defaultdict(list);grouped_pred=defaultdict(list)
    for a in gt['annotations']:grouped_gt[a['image_id']].append(a)
    for a in pred:grouped_pred[a['image_id']].append(a)
    new_gt=[];new_pred=[]
    for image in gt['images']:
        for source,dest,is_pred in [(grouped_gt,new_gt,False),(grouped_pred,new_pred,True)]:
            records,logs=merge_records(source[image['id']],image,is_pred);dest.extend(records)
            merge_logs.extend(dict(domain=domain,**r) for r in logs)
    for i,a in enumerate(new_gt,start=1):a['id']=i
    merged[domain]=(new_gt,new_pred)
merge_logs=pd.DataFrame(merge_logs);merge_logs.to_csv(OUT/'merge_components.csv',index=False)
display(merge_logs.groupby(['domain','kind','class_name']).agg(components=('members','size'),original_boxes=('members','sum'),largest_component=('members','max')))

metrics={};point_rows=[];parity=[]
for group,(domain,ids) in group_images.items():
    gt,pred=datasets[domain];mg,mp=merged[domain];specs=[]
    for cls,name in enumerate(CLASSES):
        for threshold in [0.1,0.3,0.5]:specs.append((f'original_{name}_AP{int(threshold*100)}',gt['annotations'],pred,[cls],threshold,False))
        if cls<3:specs.append((f'merged_{name}_AP50',mg,mp,[cls],0.5,False))
    for name,classes in [('all',[0,1,2,3]),('cracks',[0,1,2])]:
        for agnostic in [False,True]:specs.append((f'pooled_{name}_{"agnostic" if agnostic else "aware"}_AP50',gt['annotations'],pred,classes,0.5,agnostic))
    metrics[group]={}
    for name,annotations,detections,classes,threshold,agnostic in specs:
        metric=make_metric(annotations,detections,ids,classes,threshold,agnostic);metrics[group][name]=metric
        point_rows.append(dict(group=group,metric=name,AP=average_precision(metric),reference_instances=int(metric['ngt'].sum())))
    # Verify all original class/IoU values against a standard evaluator.
    with contextlib.redirect_stdout(io.StringIO()):
        coco=COCO();coco.dataset=gt;coco.createIndex();evaluation=COCOeval(coco,coco.loadRes(pred),'bbox')
        evaluation.params.imgIds=ids;evaluation.params.iouThrs=np.array([0.1,0.3,0.5]);evaluation.evaluate();evaluation.accumulate()
    precision=evaluation.eval['precision']
    for ci,name in enumerate(CLASSES):
        for ti,threshold in enumerate([0.1,0.3,0.5]):
            values=precision[ti,:,ci,0,2];values=values[values>=0]
            official=float(values.mean()) if len(values) else float('nan')
            own=average_precision(metrics[group][f'original_{name}_AP{int(threshold*100)}'])
            parity.append(dict(group=group,class_name=name,iou=threshold,standard=official,implementation=own,absolute_error=abs(official-own)))
    print('Scored and verified:',group,flush=True)
points=pd.DataFrame(point_rows);points.to_csv(OUT/'point_estimates.csv',index=False)
parity=pd.DataFrame(parity);parity.to_csv(OUT/'evaluator_parity.csv',index=False);assert parity.absolute_error.dropna().max()<1e-9
print('Standard evaluator parity passed. Maximum absolute error:',parity.absolute_error.max())
display(points)

# Reproduce the stored COCO baseline before interpreting alternate definitions.
stored=pd.read_csv(BASE/'localization_check/class_results.csv')
for domain in ['in_domain_dev','china_dev']:
    for name in CLASSES:
        expected=float(stored[(stored.domain==domain)&(stored['mode']=='full_frame')&(stored.class_name==name)].AP50.iloc[0])
        actual=float(points[(points.group==domain)&(points.metric==f'original_{name}_AP50')].AP.iloc[0])
        assert abs(expected-actual)<1e-9,(domain,name,expected,actual)
print('Saved full-frame COCO AP50 reproduced for every class/domain.')


Hand-checked matching and merge cases passed.


components  original_boxes  \
domain        kind       class_name                               
china_dev     GT         D00               2093            2678   
                         D10               1039            1096   
                         D20                555             641   
              prediction D00               2560           29973   
                         D10               2690           23036   
                         D20               1643           13183   
in_domain_dev GT         D00                331             348   
                         D10                234             250   
                         D20                426             451   
              prediction D00               1342            8585   
                         D10               1311            8828   
                         D20                746            7840   

                                     largest_component  
domain        kind       class_name                     
china_dev     GT         D00                         6  
                         D10                         3  
                         D20                         3  
              prediction D00                        73  
                         D10                        56  
                         D20                        51  
in_domain_dev GT         D00                         3  
                         D10                         4  
                         D20                         3  
              prediction D00                        71  
                         D10                        84  
                         D20                        75

Scored and verified: in_domain_dev
Scored and verified: Czech
Scored and verified: India
Scored and verified: Japan
Scored and verified: china_dev
Standard evaluator parity passed. Maximum absolute error: 0.0


,group,metric,AP,reference_instances
0,in_domain_dev,original_D00_AP10,0.560014,348
1,in_domain_dev,original_D00_AP30,0.509987,348
2,in_domain_dev,original_D00_AP50,0.426628,348
3,in_domain_dev,merged_D00_AP50,0.070916,331
4,in_domain_dev,original_D10_AP10,0.496669,250
...,...,...,...,...
90,china_dev,original_D40_AP50,0.023008,235
91,china_dev,pooled_all_aware_AP50,0.250854,4650
92,china_dev,pooled_all_agnostic_AP50,0.278773,4650
93,china_dev,pooled_cracks_aware_AP50,0.276895,4415


Saved full-frame COCO AP50 reproduced for every class/domain.


## 4. Paired image bootstrap with fixed seed
Use 1,000 image-level bootstrap samples per group. Reuse the SAME sampled image weights for original, merged and class-agnostic scores so differences are paired. Save percentile 95% intervals for AP and the paired changes. A two-percentage-point rule is only a practical heuristic: inspect the interval rather than automatically declaring every smaller change noise. These intervals are conditional on the fixed model and cached predictions; they do not measure training-seed variability. Adjacent road frames may be correlated, so independent-image intervals may be optimistic. Empty-reference replicates are excluded and counted. A 15-minute bootstrap budget saves partial work and stops with an explicit error rather than publishing incomplete intervals.


In [4]:
bootstrap_start=time.monotonic();ci_rows=[];delta_rows=[]
replicates={};rng=np.random.default_rng(protocol['bootstrap_seed'])
for group,group_metrics in metrics.items():
    n=len(next(iter(group_metrics.values()))['ngt'])
    samples={name:np.empty(protocol['bootstrap_replicates']) for name in group_metrics}
    for b in range(protocol['bootstrap_replicates']):
        weights=np.bincount(rng.integers(0,n,size=n),minlength=n)
        for name,metric in group_metrics.items():samples[name][b]=average_precision(metric,weights)
        if (b+1)%100==0:
            print(group,'bootstrap',b+1,'/',protocol['bootstrap_replicates'],'elapsed',round(time.monotonic()-bootstrap_start,1),flush=True)
            assert time.monotonic()-bootstrap_start<900,'Bootstrap budget reached; do not interpret partial intervals.'
    replicates[group]=samples
    for name,values in samples.items():
        valid=values[np.isfinite(values)];lo,hi=np.quantile(valid,[0.025,0.975]) if len(valid) else (np.nan,np.nan)
        ci_rows.append(dict(group=group,metric=name,AP=average_precision(group_metrics[name]),CI_low=lo,CI_high=hi,valid_replicates=len(valid)))
    pairs=[]
    for name in CLASSES[:3]:pairs.append((f'merged_minus_original_{name}_AP50',f'merged_{name}_AP50',f'original_{name}_AP50'))
    for name in CLASSES:pairs.append((f'AP10_minus_AP50_{name}',f'original_{name}_AP10',f'original_{name}_AP50'))
    for name in ['all','cracks']:pairs.append((f'agnostic_minus_aware_{name}_AP50',f'pooled_{name}_agnostic_AP50',f'pooled_{name}_aware_AP50'))
    for label,left,right in pairs:
        values=samples[left]-samples[right];valid=values[np.isfinite(values)]
        lo,hi=np.quantile(valid,[0.025,0.975]) if len(valid) else (np.nan,np.nan)
        delta_rows.append(dict(group=group,comparison=label,delta=average_precision(group_metrics[left])-average_precision(group_metrics[right]),CI_low=lo,CI_high=hi,valid_replicates=len(valid)))
    pd.DataFrame(ci_rows).to_csv(OUT/'bootstrap_AP.csv',index=False);pd.DataFrame(delta_rows).to_csv(OUT/'bootstrap_differences.csv',index=False)
np.savez_compressed(OUT/'bootstrap_replicates.npz',**{group+'__'+name:values for group,samples in replicates.items() for name,values in samples.items()})
confidence=pd.DataFrame(ci_rows);differences=pd.DataFrame(delta_rows);display(confidence);display(differences)


in_domain_dev bootstrap 100 / 1000 elapsed 0.8
in_domain_dev bootstrap 200 / 1000 elapsed 1.5
in_domain_dev bootstrap 300 / 1000 elapsed 2.2
in_domain_dev bootstrap 400 / 1000 elapsed 3.0
in_domain_dev bootstrap 500 / 1000 elapsed 3.7
in_domain_dev bootstrap 600 / 1000 elapsed 4.4
in_domain_dev bootstrap 700 / 1000 elapsed 5.2
in_domain_dev bootstrap 800 / 1000 elapsed 5.9
in_domain_dev bootstrap 900 / 1000 elapsed 6.6
in_domain_dev bootstrap 1000 / 1000 elapsed 7.4
Czech bootstrap 100 / 1000 elapsed 7.6
Czech bootstrap 200 / 1000 elapsed 7.8
Czech bootstrap 300 / 1000 elapsed 7.9
Czech bootstrap 400 / 1000 elapsed 8.1
Czech bootstrap 500 / 1000 elapsed 8.3
Czech bootstrap 600 / 1000 elapsed 8.5
Czech bootstrap 700 / 1000 elapsed 8.6
Czech bootstrap 800 / 1000 elapsed 8.8
Czech bootstrap 900 / 1000 elapsed 9.0
Czech bootstrap 1000 / 1000 elapsed 9.2
India bootstrap 100 / 1000 elapsed 9.3
India bootstrap 200 / 1000 elapsed 9.5
India bootstrap 300 / 1000 elapsed 9.6
India bootstrap 400 /

,group,metric,AP,CI_low,CI_high,valid_replicates
0,in_domain_dev,original_D00_AP10,0.560014,0.509186,0.617785,1000
1,in_domain_dev,original_D00_AP30,0.509987,0.457300,0.570475,1000
2,in_domain_dev,original_D00_AP50,0.426628,0.370889,0.485456,1000
3,in_domain_dev,merged_D00_AP50,0.070916,0.045614,0.108905,1000
4,in_domain_dev,original_D10_AP10,0.496669,0.428650,0.570522,1000
...,...,...,...,...,...,...
90,china_dev,original_D40_AP50,0.023008,0.014807,0.038137,1000
91,china_dev,pooled_all_aware_AP50,0.250854,0.235954,0.267815,1000
92,china_dev,pooled_all_agnostic_AP50,0.278773,0.263692,0.295693,1000
93,china_dev,pooled_cracks_aware_AP50,0.276895,0.261612,0.294227,1000


,group,comparison,delta,CI_low,CI_high,valid_replicates
0,in_domain_dev,merged_minus_original_D00_AP50,-0.355711,-0.412349,-0.294585,1000
1,in_domain_dev,merged_minus_original_D10_AP50,-0.326381,-0.394241,-0.256463,1000
2,in_domain_dev,merged_minus_original_D20_AP50,-0.401620,-0.453366,-0.336093,1000
3,in_domain_dev,AP10_minus_AP50_D00,0.133387,0.098408,0.170661,1000
4,in_domain_dev,AP10_minus_AP50_D10,0.134933,0.091579,0.181016,1000
5,in_domain_dev,AP10_minus_AP50_D20,0.130234,0.098836,0.164272,1000
6,in_domain_dev,AP10_minus_AP50_D40,0.114223,0.071186,0.157084,1000
7,in_domain_dev,agnostic_minus_aware_all_AP50,0.021302,0.012992,0.029990,1000
8,in_domain_dev,agnostic_minus_aware_cracks_AP50,0.023305,0.013178,0.033567,1000
9,Czech,merged_minus_original_D00_AP50,-0.304487,-0.431566,-0.179435,1000


## 5. Apply the declared checks and preserve evidence
Report the proposed 0.30/0.03/0.10 thresholds as provisional diagnostic checks, with paired intervals and country support. Lower IoU and merged boxes change the task definition; a higher score cannot establish genuine detection improvement. Similar scores across domains do not rule out generalization failure. Freeze the protocol, source hashes, component memberships, baseline parity and bootstrap draws. Leave weights and source annotations untouched; no retraining or relabeling follows automatically.


In [6]:
def value(group,name):return float(points[(points.group==group)&(points.metric==name)].AP.iloc[0])
def difference(group,name):return differences[(differences.group==group)&(differences.comparison==name)].iloc[0]
china_merged=value('china_dev','merged_D00_AP50')
control=abs(value('china_dev','merged_D20_AP50')-value('china_dev','original_D20_AP50'))
country_improvements={country:value(country,'merged_D00_AP50')-value(country,'original_D00_AP50') for country in ['Czech','India','Japan']}
checks=dict(China_D00_merged_at_least_030=china_merged>=0.30,D20_control_change_at_most_003=control<=0.03,D00_point_improvement_in_each_country=all(d>0 for d in country_improvements.values()),China_D00_AP10_below_030=value('china_dev','original_D00_AP10')<0.30,pooled_crack_class_gap_above_010=difference('china_dev','agnostic_minus_aware_cracks_AP50')['delta']>0.10)
checks={key:bool(result) for key,result in checks.items()}
(OUT/'diagnostic_checks.json').write_text(json.dumps(checks,indent=2))
print('Provisional diagnostic checks:',json.dumps(checks,indent=2))
original=confidence[confidence.metric.str.startswith('original_')].pivot(index='group',columns='metric',values='AP')
merged_table=confidence[confidence.metric.str.startswith('merged_')]
notes="""No model or source label was changed. These are sensitivity analyses of fixed cached predictions, not new performance claims. The cache was capped at 100 detections/image and used batch two; its COCO baseline differs from earlier Ultralytics cap-300 validation. Fragments may join distinct damage regions, and connected low-confidence predictions may form large boxes. Review merge component sizes before interpreting merged AP as annotation evidence. Higher AP at low IoU only indicates coarse overlap; it cannot verify accurate crack localization. Class-agnostic comparisons use paired pooled class-aware baselines, not macro AP. Image bootstrap does not capture training randomness and may underestimate uncertainty for correlated adjacent frames. The prespecified checks are provisional and do not justify automatic relabeling or assert causation. Development data remain reused, not blind. No test data, training, GPU inference or original-image downloads were needed."""
report=['# RoadScan fixed-prediction re-scoring audit','## Registered protocol',json.dumps(protocol,indent=2),'## Reference support',support.to_markdown(index=False),'## Exact monitor composition',monitor.to_markdown(index=False),'## Original AP at several IoU thresholds',original.to_markdown(),'## Merged AP with 95% image-bootstrap intervals',merged_table.to_markdown(index=False),'## Paired differences and 95% intervals',differences.to_markdown(index=False),'## Provisional checks',json.dumps(checks,indent=2),'D00 merged changes by country: '+json.dumps(country_improvements),'## Interpretation limits',notes,'## Timing','Total CPU analysis seconds: '+str(round(time.monotonic()-start,1))]
(OUT/'REPORT.md').write_text('\n\n'.join(report))
(OUT/'OUTPUT_SHA256.json').write_text(json.dumps({p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.iterdir() if p.is_file() and p.name!='OUTPUT_SHA256.json'},indent=2))
for name,digest in json.loads((OUT/'OUTPUT_SHA256.json').read_text()).items():assert hashlib.sha256((OUT/name).read_bytes()).hexdigest()==digest
import shutil
shutil.make_archive('/kaggle/working/roadscan_rescoring_audit','zip',root_dir=OUT)
display(Markdown('\n\n'.join(report)))
print('RE-SCORING COMPLETE. CPU only. Source files unchanged; all output hashes verified.')

china_table=confidence[confidence.group=='china_dev'].copy()
china_table=china_table[china_table.metric.str.startswith(('original_','merged_'))]
country_d00=confidence[(confidence.group.isin(['Czech','India','Japan'])) & (confidence.metric.isin(['original_D00_AP50','merged_D00_AP50']))]
class_gap=differences[differences.comparison.str.startswith('agnostic_minus_aware')]
print('CHINA CLASS SCORES AND 95% INTERVALS');display(china_table)
print('D00 COUNTRY COMPARISON');display(country_d00)
print('PAIRED CLASS-CONFUSION GAPS');display(class_gap)
interpretation='''## Scientific interpretation

The proposed fragment-merging success criterion was not met. Merging lowered China D00 AP50, and D20 changed substantially, so the proposed control condition failed. Merging also lowered D00 AP50 in each source country. This result rejects this particular fixed 2% connected-component scoring rule as a useful correction under the saved low-confidence prediction protocol. It does NOT establish that source annotations are semantically correct or that fragment conventions are irrelevant: merging creates large prediction chains, up to 73 D00 boxes in a China component, while the largest reference component has six boxes. The GT and prediction geometries respond very differently to the same rule. Do not relabel the dataset using this rule.

Relaxing IoU from 0.50 to 0.10 increases China D00 AP by about 0.272 (27.2 percentage points), with a paired image-bootstrap interval excluding zero. The AP10 value exceeds 0.30, so the proposed low-overlap failure criterion for D00 is not met. This supports localization/extent sensitivity as one contributor, but IoU 0.10 is permissive and cannot prove accurate crack detection or annotation inconsistency. D10 and D20 also improve under lower overlap requirements.

The China pooled crack class-agnostic versus class-aware AP50 gap is about 0.023 (2.3 percentage points), well below the proposed 0.10 threshold, though its paired interval excludes zero. Classification mistakes exist but this test does not identify them as the dominant pooled cause. This is a pooled comparison, not a comparison against macro class AP.

D40 remains weak in China even at IoU 0.10, suggesting additional missed-detection, appearance, scale, confidence or annotation factors. A box-only re-scoring experiment cannot distinguish these causes. D40 has 235 reference boxes in 164 images, so its poor score is not based on one or two instances.

The monitor composition is known: 200 images from each development domain, with 256 in-domain boxes and 459 China boxes. Its combined score need not match the larger development set because composition and support differ. Country-level evidence is uneven: India D10 has only one positive image and one reference box; only 641 of 1,000 bootstrap draws contain this class. Such a result cannot support a strong country-specific generalization claim.

Next decision: keep official metrics and weights unchanged. Prioritize a small, blinded expert review of original crack boxes beside fixed predictions to distinguish partial localization, missed segments, multiple nearby defects and semantic label disagreements. Review D40 separately. Any future label revision needs an explicit annotation rule and independent review; do not merge source boxes automatically or train on development/test images. Do not start another long training run from these diagnostics alone.
'''
with (OUT/'REPORT.md').open('a') as file:
    file.write('\n\n## China class scores and intervals\n\n'+china_table.to_markdown(index=False)+'\n\n## D00 country comparison\n\n'+country_d00.to_markdown(index=False)+'\n\n'+interpretation)
(OUT/'INTERPRETATION.md').write_text(interpretation)
china_table.to_csv(OUT/'china_summary.csv',index=False);country_d00.to_csv(OUT/'country_D00_summary.csv',index=False)
(OUT/'OUTPUT_SHA256.json').write_text(json.dumps({p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.iterdir() if p.is_file() and p.name!='OUTPUT_SHA256.json'},indent=2))
for name,digest in json.loads((OUT/'OUTPUT_SHA256.json').read_text()).items():assert hashlib.sha256((OUT/name).read_bytes()).hexdigest()==digest
shutil.make_archive('/kaggle/working/roadscan_rescoring_audit','zip',root_dir=OUT)
display(Markdown(interpretation))
print('FINAL RE-SCORING PACKAGE VERIFIED. CPU session can be stopped after output snapshot is verified.')


Provisional diagnostic checks: {
  "China_D00_merged_at_least_030": false,
  "D20_control_change_at_most_003": false,
  "D00_point_improvement_in_each_country": false,
  "China_D00_AP10_below_030": false,
  "pooled_crack_class_gap_above_010": false
}


# RoadScan fixed-prediction re-scoring audit

## Registered protocol

{
  "source_notebook_version": 359330902,
  "weights": "notebook 10 selected checkpoint; notebook 11 unchanged",
  "imgsz": 640,
  "conf_floor": 0.001,
  "nms_iou": 0.7,
  "saved_max_det": 100,
  "inference_batch": 2,
  "bootstrap_replicates": 1000,
  "bootstrap_seed": 20261017,
  "merge_gap_fraction": 0.02,
  "merge_score": "maximum member confidence",
  "merge_link": "connected components of ORIGINAL same-class rectangles; normalized x/y gaps each <=0.02",
  "thresholds": [
    0.1,
    0.3,
    0.5
  ],
  "criterion_D00_merged_China": 0.3,
  "criterion_control_D20_max_abs_change": 0.03,
  "criterion_D00_AP10_China": 0.3,
  "criterion_pooled_class_gap": 0.1
}

## Reference support

| group         | class_name   |   images |   positive_images |   instances |   background_images |
|:--------------|:-------------|---------:|------------------:|------------:|--------------------:|
| in_domain_dev | D00          |     1000 |               250 |         348 |                 358 |
| in_domain_dev | D10          |     1000 |               153 |         250 |                 358 |
| in_domain_dev | D20          |     1000 |               349 |         451 |                 358 |
| in_domain_dev | D40          |     1000 |               126 |         211 |                 358 |
| Czech         | D00          |      202 |                50 |          62 |                 133 |
| Czech         | D10          |      202 |                13 |          17 |                 133 |
| Czech         | D20          |      202 |                10 |          14 |                 133 |
| Czech         | D40          |      202 |                13 |          17 |                 133 |
| India         | D00          |      124 |                15 |          17 |                  67 |
| India         | D10          |      124 |                 1 |           1 |                  67 |
| India         | D20          |      124 |                36 |          40 |                  67 |
| India         | D40          |      124 |                24 |          50 |                  67 |
| Japan         | D00          |      674 |               185 |         269 |                 158 |
| Japan         | D10          |      674 |               139 |         232 |                 158 |
| Japan         | D20          |      674 |               303 |         397 |                 158 |
| Japan         | D40          |      674 |                89 |         144 |                 158 |
| china_dev     | D00          |     1977 |              1419 |        2678 |                  43 |
| china_dev     | D10          |     1977 |               738 |        1096 |                  43 |
| china_dev     | D20          |     1977 |               486 |         641 |                  43 |
| china_dev     | D40          |     1977 |               164 |         235 |                  43 |

## Exact monitor composition

| domain        |   images |   backgrounds |   instances |
|:--------------|---------:|--------------:|------------:|
| in_domain_dev |      200 |            65 |         256 |
| china_dev     |      200 |             6 |         459 |

## Original AP at several IoU thresholds

| group         |   original_D00_AP10 |   original_D00_AP30 |   original_D00_AP50 |   original_D10_AP10 |   original_D10_AP30 |   original_D10_AP50 |   original_D20_AP10 |   original_D20_AP30 |   original_D20_AP50 |   original_D40_AP10 |   original_D40_AP30 |   original_D40_AP50 |
|:--------------|--------------------:|--------------------:|--------------------:|--------------------:|--------------------:|--------------------:|--------------------:|--------------------:|--------------------:|--------------------:|--------------------:|--------------------:|
| Czech         |            0.529651 |            0.488212 |            0.385082 |            0.448946 |            0.412694 |            0.350524 |            0.609507 |            0.60581  |            0.507839 |            0.201101 |           0.197657  |           0.158297  |
| India         |            0.531176 |            0.516358 |            0.506629 |            0        |            0        |            0        |            0.859477 |            0.805626 |            0.734283 |            0.558918 |           0.48761   |           0.392987  |
| Japan         |            0.572691 |            0.520058 |            0.441145 |            0.507702 |            0.45462  |            0.369362 |            0.824217 |            0.776909 |            0.693186 |            0.670583 |           0.646086  |           0.573246  |
| china_dev     |            0.465839 |            0.343305 |            0.193658 |            0.683186 |            0.571883 |            0.360017 |            0.625969 |            0.561446 |            0.467751 |            0.169091 |           0.0755581 |           0.0230079 |
| in_domain_dev |            0.560014 |            0.509987 |            0.426628 |            0.496669 |            0.445476 |            0.361735 |            0.819988 |            0.773223 |            0.689754 |            0.608856 |           0.575281  |           0.494634  |

## Merged AP with 95% image-bootstrap intervals

| group         | metric          |        AP |     CI_low |   CI_high |   valid_replicates |
|:--------------|:----------------|----------:|-----------:|----------:|-------------------:|
| in_domain_dev | merged_D00_AP50 | 0.0709163 | 0.0456139  | 0.108905  |               1000 |
| in_domain_dev | merged_D10_AP50 | 0.035354  | 0.0201206  | 0.0656949 |               1000 |
| in_domain_dev | merged_D20_AP50 | 0.288134  | 0.23486    | 0.35334   |               1000 |
| Czech         | merged_D00_AP50 | 0.0805948 | 0.0237008  | 0.16467   |               1000 |
| Czech         | merged_D10_AP50 | 0.0395916 | 0.00660066 | 0.15497   |               1000 |
| Czech         | merged_D20_AP50 | 0.0968012 | 0          | 0.355898  |               1000 |
| India         | merged_D00_AP50 | 0.0783343 | 0.00424033 | 0.331541  |               1000 |
| India         | merged_D10_AP50 | 0         | 0          | 0         |                641 |
| India         | merged_D20_AP50 | 0.522653  | 0.337446   | 0.72077   |               1000 |
| Japan         | merged_D00_AP50 | 0.0746236 | 0.0461144  | 0.119412  |               1000 |
| Japan         | merged_D10_AP50 | 0.0353744 | 0.0204024  | 0.066107  |               1000 |
| Japan         | merged_D20_AP50 | 0.280184  | 0.22568    | 0.348055  |               1000 |
| china_dev     | merged_D00_AP50 | 0.105724  | 0.0910494  | 0.125369  |               1000 |
| china_dev     | merged_D10_AP50 | 0.0454562 | 0.0343055  | 0.0562709 |               1000 |
| china_dev     | merged_D20_AP50 | 0.275422  | 0.232328   | 0.31897   |               1000 |

## Paired differences and 95% intervals

| group         | comparison                       |       delta |       CI_low |     CI_high |   valid_replicates |
|:--------------|:---------------------------------|------------:|-------------:|------------:|-------------------:|
| in_domain_dev | merged_minus_original_D00_AP50   | -0.355711   | -0.412349    | -0.294585   |               1000 |
| in_domain_dev | merged_minus_original_D10_AP50   | -0.326381   | -0.394241    | -0.256463   |               1000 |
| in_domain_dev | merged_minus_original_D20_AP50   | -0.40162    | -0.453366    | -0.336093   |               1000 |
| in_domain_dev | AP10_minus_AP50_D00              |  0.133387   |  0.0984076   |  0.170661   |               1000 |
| in_domain_dev | AP10_minus_AP50_D10              |  0.134933   |  0.091579    |  0.181016   |               1000 |
| in_domain_dev | AP10_minus_AP50_D20              |  0.130234   |  0.0988361   |  0.164272   |               1000 |
| in_domain_dev | AP10_minus_AP50_D40              |  0.114223   |  0.0711857   |  0.157084   |               1000 |
| in_domain_dev | agnostic_minus_aware_all_AP50    |  0.021302   |  0.0129917   |  0.0299901  |               1000 |
| in_domain_dev | agnostic_minus_aware_cracks_AP50 |  0.0233048  |  0.0131783   |  0.0335671  |               1000 |
| Czech         | merged_minus_original_D00_AP50   | -0.304487   | -0.431566    | -0.179435   |               1000 |
| Czech         | merged_minus_original_D10_AP50   | -0.310932   | -0.519557    | -0.0564867  |               1000 |
| Czech         | merged_minus_original_D20_AP50   | -0.411038   | -0.71998     |  0.0477933  |               1000 |
| Czech         | AP10_minus_AP50_D00              |  0.144569   |  0.0746228   |  0.234189   |               1000 |
| Czech         | AP10_minus_AP50_D10              |  0.0984224  |  0.00829568  |  0.218916   |               1000 |
| Czech         | AP10_minus_AP50_D20              |  0.101668   |  0           |  0.337398   |               1000 |
| Czech         | AP10_minus_AP50_D40              |  0.0428041  |  0.00380911  |  0.133771   |               1000 |
| Czech         | agnostic_minus_aware_all_AP50    |  0.00947038 |  0.00175693  |  0.0181411  |               1000 |
| Czech         | agnostic_minus_aware_cracks_AP50 |  0.00301043 |  0           |  0.00748007 |               1000 |
| India         | merged_minus_original_D00_AP50   | -0.428295   | -0.623201    | -0.17608    |               1000 |
| India         | merged_minus_original_D10_AP50   |  0          |  0           |  0          |                641 |
| India         | merged_minus_original_D20_AP50   | -0.211631   | -0.424966    |  0.00847251 |               1000 |
| India         | AP10_minus_AP50_D00              |  0.0245476  |  0.00178337  |  0.0567772  |               1000 |
| India         | AP10_minus_AP50_D10              |  0          |  0           |  0          |                641 |
| India         | AP10_minus_AP50_D20              |  0.125193   |  0.0287906   |  0.23238    |               1000 |
| India         | AP10_minus_AP50_D40              |  0.16593    |  0.06007     |  0.277497   |               1000 |
| India         | agnostic_minus_aware_all_AP50    |  0.00886777 |  0.000714308 |  0.0225518  |               1000 |
| India         | agnostic_minus_aware_cracks_AP50 |  0.00593064 |  0           |  0.0200084  |               1000 |
| Japan         | merged_minus_original_D00_AP50   | -0.366521   | -0.441568    | -0.298979   |               1000 |
| Japan         | merged_minus_original_D10_AP50   | -0.333988   | -0.400203    | -0.257657   |               1000 |
| Japan         | merged_minus_original_D20_AP50   | -0.413002   | -0.474645    | -0.342622   |               1000 |
| Japan         | AP10_minus_AP50_D00              |  0.131546   |  0.0873777   |  0.174875   |               1000 |
| Japan         | AP10_minus_AP50_D10              |  0.13834    |  0.0952145   |  0.18855    |               1000 |
| Japan         | AP10_minus_AP50_D20              |  0.131031   |  0.0959136   |  0.167349   |               1000 |
| Japan         | AP10_minus_AP50_D40              |  0.0973365  |  0.0568542   |  0.144069   |               1000 |
| Japan         | agnostic_minus_aware_all_AP50    |  0.0233917  |  0.0142537   |  0.0335466  |               1000 |
| Japan         | agnostic_minus_aware_cracks_AP50 |  0.02592    |  0.015352    |  0.0374446  |               1000 |
| china_dev     | merged_minus_original_D00_AP50   | -0.0879337  | -0.113445    | -0.0626013  |               1000 |
| china_dev     | merged_minus_original_D10_AP50   | -0.314561   | -0.348329    | -0.282514   |               1000 |
| china_dev     | merged_minus_original_D20_AP50   | -0.192329   | -0.233323    | -0.152931   |               1000 |
| china_dev     | AP10_minus_AP50_D00              |  0.272182   |  0.257254    |  0.287077   |               1000 |
| china_dev     | AP10_minus_AP50_D10              |  0.323169   |  0.295172    |  0.353647   |               1000 |
| china_dev     | AP10_minus_AP50_D20              |  0.158218   |  0.128992    |  0.184477   |               1000 |
| china_dev     | AP10_minus_AP50_D40              |  0.146083   |  0.103459    |  0.198033   |               1000 |
| china_dev     | agnostic_minus_aware_all_AP50    |  0.0279186  |  0.0236056   |  0.0324836  |               1000 |
| china_dev     | agnostic_minus_aware_cracks_AP50 |  0.0230313  |  0.0190035   |  0.027344   |               1000 |

## Provisional checks

{
  "China_D00_merged_at_least_030": false,
  "D20_control_change_at_most_003": false,
  "D00_point_improvement_in_each_country": false,
  "China_D00_AP10_below_030": false,
  "pooled_crack_class_gap_above_010": false
}

D00 merged changes by country: {"Czech": -0.3044872487124241, "India": -0.42829454896734664, "Japan": -0.36652120669143623}

## Interpretation limits

No model or source label was changed. These are sensitivity analyses of fixed cached predictions, not new performance claims. The cache was capped at 100 detections/image and used batch two; its COCO baseline differs from earlier Ultralytics cap-300 validation. Fragments may join distinct damage regions, and connected low-confidence predictions may form large boxes. Review merge component sizes before interpreting merged AP as annotation evidence. Higher AP at low IoU only indicates coarse overlap; it cannot verify accurate crack localization. Class-agnostic comparisons use paired pooled class-aware baselines, not macro AP. Image bootstrap does not capture training randomness and may underestimate uncertainty for correlated adjacent frames. The prespecified checks are provisional and do not justify automatic relabeling or assert causation. Development data remain reused, not blind. No test data, training, GPU inference or original-image downloads were needed.

## Timing

Total CPU analysis seconds: 209.4

RE-SCORING COMPLETE. CPU only. Source files unchanged; all output hashes verified.
CHINA CLASS SCORES AND 95% INTERVALS


,group,metric,AP,CI_low,CI_high,valid_replicates
76,china_dev,original_D00_AP10,0.465839,0.448750,0.486429,1000
77,china_dev,original_D00_AP30,0.343305,0.324073,0.367549,1000
78,china_dev,original_D00_AP50,0.193658,0.177867,0.214199,1000
79,china_dev,merged_D00_AP50,0.105724,0.091049,0.125369,1000
80,china_dev,original_D10_AP10,0.683186,0.662187,0.708588,1000
81,china_dev,original_D10_AP30,0.571883,0.546512,0.602425,1000
82,china_dev,original_D10_AP50,0.360017,0.327750,0.392252,1000
83,china_dev,merged_D10_AP50,0.045456,0.034305,0.056271,1000
84,china_dev,original_D20_AP10,0.625969,0.593978,0.658308,1000
85,china_dev,original_D20_AP30,0.561446,0.525689,0.598295,1000


D00 COUNTRY COMPARISON


,group,metric,AP,CI_low,CI_high,valid_replicates
21,Czech,original_D00_AP50,0.385082,0.266515,0.523353,1000
22,Czech,merged_D00_AP50,0.080595,0.023701,0.164670,1000
40,India,original_D00_AP50,0.506629,0.299012,0.737574,1000
41,India,merged_D00_AP50,0.078334,0.004240,0.331541,1000
59,Japan,original_D00_AP50,0.441145,0.384296,0.518592,1000
60,Japan,merged_D00_AP50,0.074624,0.046114,0.119412,1000


PAIRED CLASS-CONFUSION GAPS


,group,comparison,delta,CI_low,CI_high,valid_replicates
7,in_domain_dev,agnostic_minus_aware_all_AP50,0.021302,0.012992,0.029990,1000
8,in_domain_dev,agnostic_minus_aware_cracks_AP50,0.023305,0.013178,0.033567,1000
16,Czech,agnostic_minus_aware_all_AP50,0.009470,0.001757,0.018141,1000
17,Czech,agnostic_minus_aware_cracks_AP50,0.003010,0.000000,0.007480,1000
25,India,agnostic_minus_aware_all_AP50,0.008868,0.000714,0.022552,1000
26,India,agnostic_minus_aware_cracks_AP50,0.005931,0.000000,0.020008,1000
34,Japan,agnostic_minus_aware_all_AP50,0.023392,0.014254,0.033547,1000
35,Japan,agnostic_minus_aware_cracks_AP50,0.025920,0.015352,0.037445,1000
43,china_dev,agnostic_minus_aware_all_AP50,0.027919,0.023606,0.032484,1000
44,china_dev,agnostic_minus_aware_cracks_AP50,0.023031,0.019003,0.027344,1000


## Scientific interpretation

The proposed fragment-merging success criterion was not met. Merging lowered China D00 AP50, and D20 changed substantially, so the proposed control condition failed. Merging also lowered D00 AP50 in each source country. This result rejects this particular fixed 2% connected-component scoring rule as a useful correction under the saved low-confidence prediction protocol. It does NOT establish that source annotations are semantically correct or that fragment conventions are irrelevant: merging creates large prediction chains, up to 73 D00 boxes in a China component, while the largest reference component has six boxes. The GT and prediction geometries respond very differently to the same rule. Do not relabel the dataset using this rule.

Relaxing IoU from 0.50 to 0.10 increases China D00 AP by about 0.272 (27.2 percentage points), with a paired image-bootstrap interval excluding zero. The AP10 value exceeds 0.30, so the proposed low-overlap failure criterion for D00 is not met. This supports localization/extent sensitivity as one contributor, but IoU 0.10 is permissive and cannot prove accurate crack detection or annotation inconsistency. D10 and D20 also improve under lower overlap requirements.

The China pooled crack class-agnostic versus class-aware AP50 gap is about 0.023 (2.3 percentage points), well below the proposed 0.10 threshold, though its paired interval excludes zero. Classification mistakes exist but this test does not identify them as the dominant pooled cause. This is a pooled comparison, not a comparison against macro class AP.

D40 remains weak in China even at IoU 0.10, suggesting additional missed-detection, appearance, scale, confidence or annotation factors. A box-only re-scoring experiment cannot distinguish these causes. D40 has 235 reference boxes in 164 images, so its poor score is not based on one or two instances.

The monitor composition is known: 200 images from each development domain, with 256 in-domain boxes and 459 China boxes. Its combined score need not match the larger development set because composition and support differ. Country-level evidence is uneven: India D10 has only one positive image and one reference box; only 641 of 1,000 bootstrap draws contain this class. Such a result cannot support a strong country-specific generalization claim.

Next decision: keep official metrics and weights unchanged. Prioritize a small, blinded expert review of original crack boxes beside fixed predictions to distinguish partial localization, missed segments, multiple nearby defects and semantic label disagreements. Review D40 separately. Any future label revision needs an explicit annotation rule and independent review; do not merge source boxes automatically or train on development/test images. Do not start another long training run from these diagnostics alone.


FINAL RE-SCORING PACKAGE VERIFIED. CPU session can be stopped after output snapshot is verified.
